In [7]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
short_metrics_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "faithfulness_estimate": "Faith.Est",
    "monotonicity_correlation": "Mono.Corr",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}
explainers = ["SoftPullback", "PullbackAscent", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]
# explainers = ["SoftPullback", "PullbackAscent", "PullbackAscentNoAlpha", "PullbackAscent3", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]

In [9]:
def merge_with_priority(df_big, df_small, key_col="Faith.Corr"):
    # we merge results as we re-genenerated the faithfulness_correlation metric for different set of patches
    # the previous selection had too large patches, which inflated the correlation score
    result = df_big.loc[df_small.index].copy()
    result[key_col] = df_small[key_col]
    return result

def process_df(df_path, precision=3):
    selected_columns = list(short_metrics_map.keys())
    
    df = QuantusEvaluator.load_results(df_path)
    # df = df[selected_columns].rename(index=short_metrics_map, columns=short_metrics_map)
    df = df[
        [col for col in selected_columns if col in df.columns]
    ].rename(index=short_metrics_map, columns=short_metrics_map)
    # df = df.loc[explainers]
    df = df.loc[[idx for idx in explainers if idx in df.index]]
    
    print("num_samples:", len(df.iloc[0].iloc[0]))
    
    df = QuantusEvaluator.summarize_results(df, precision=precision)
    return df


In [10]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn")
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,16.303±31.68,0.606±0.3,0.53±0.4,0.375±0.39,0.212±0.08,-0.064±0.33
PullbackAscent,5.616±13.18,0.57±0.29,0.376±0.48,0.312±0.37,0.39±0.11,0.136±0.13
SmoothPullback,14.03±30.57,0.545±0.32,0.428±0.46,0.309±0.4,0.398±0.11,-0.048±0.32
FusionPullback,16.056±34.46,0.513±0.33,0.406±0.47,0.315±0.39,0.513±0.11,-0.035±0.29
Gradient,100.09±107.94,0.457±0.35,0.466±0.41,0.316±0.37,0.898±0.14,-0.053±0.29
GradientAscent,18.523±35.24,0.508±0.3,0.264±0.46,0.234±0.32,1.217±0.06,6.65e-04±0.07
SmoothGrad,52.044±70.8,0.513±0.34,0.508±0.41,0.385±0.36,0.781±0.11,-0.025±0.19
FusionGrad,38.651±60.23,0.506±0.34,0.521±0.39,0.363±0.34,0.955±0.13,-0.013±0.16
GradientShap,76.05±94.76,0.43±0.36,0.632±0.33,0.381±0.41,1.086±0.22,-0.042±0.26
IntegratedGradients,75.796±92.29,0.434±0.36,0.634±0.33,0.379±0.41,0.752±0.15,-0.046±0.27


In [11]:
pvt_df_fc = process_df("results/quantus_pvt_fc_nips_20_50_pvt_v2_b1")
pvt_df = process_df("results/quantus_pvt_it_nips_20_50_pvt_v2_b1")
pvt_merged_df = merge_with_priority(pvt_df, pvt_df_fc)
pvt_merged_df.to_csv('results/pvt_df_20_50.csv', index=True)
pvt_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,6.264±6.4,0.119±0.34,0.16±0.41,0.164±0.43,1.066±0.18,-0.006±0.35
PullbackAscent,1.634±1.03,0.122±0.36,0.201±0.41,0.219±0.33,0.855±0.12,0.121±0.11
SmoothPullback,4.974±5.46,0.102±0.39,0.146±0.49,0.231±0.38,0.519±0.09,0.008±0.23
FusionPullback,5.156±5.72,0.093±0.39,0.114±0.49,0.2±0.38,0.753±0.11,0.007±0.19
Gradient,8.914±7.89,0.105±0.33,0.117±0.41,0.185±0.43,1.034±0.16,-0.03±0.3
GradientAscent,4.506±4.07,0.118±0.33,0.06±0.42,0.164±0.35,1.242±0.07,0.006±0.06
SmoothGrad,8.798±7.08,0.096±0.38,0.171±0.48,0.28±0.38,0.569±0.1,-0.012±0.13
FusionGrad,6.673±5.6,0.101±0.39,0.144±0.48,0.262±0.38,0.979±0.16,5.73e-04±0.11
GradientShap,12.433±8.45,0.067±0.32,0.177±0.39,0.169±0.45,2.471±1.56,0.003±0.34
IntegratedGradients,12.578±8.39,0.068±0.32,0.157±0.4,0.17±0.45,1.374±0.37,0.008±0.33


In [12]:
resnet_df_fc = process_df("results/quantus_resnet_fc_nips_20_50_resnet50")
resnet_df = process_df("results/quantus_resnet_it_nips_20_50_resnet50")
resnet_merged_df = merge_with_priority(resnet_df, resnet_df_fc)
resnet_merged_df.to_csv('results/resnet_df_20_50.csv', index=True)
resnet_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,5.989±5.67,0.389±0.37,0.437±0.41,0.42±0.39,0.119±0.04,-0.062±0.42
PullbackAscent,5.384±4.83,0.382±0.37,0.394±0.43,0.421±0.4,0.244±0.09,0.212±0.19
SmoothPullback,8.122±10.13,0.326±0.39,0.321±0.46,0.352±0.4,0.22±0.05,-0.065±0.4
FusionPullback,10.131±14.55,0.306±0.4,0.306±0.47,0.35±0.4,0.362±0.08,-0.051±0.39
Gradient,83.294±68.15,0.241±0.36,0.289±0.45,0.275±0.39,0.952±0.17,-0.032±0.23
GradientAscent,29.48±35.36,0.278±0.37,0.131±0.46,0.231±0.32,1.267±0.04,0.002±0.05
SmoothGrad,66.883±58.16,0.331±0.4,0.321±0.49,0.364±0.36,0.639±0.08,-0.024±0.2
FusionGrad,58.514±57.8,0.344±0.39,0.336±0.48,0.357±0.35,0.855±0.09,-0.014±0.17
GradientShap,69.072±62.93,0.233±0.38,0.409±0.44,0.344±0.41,1.4±0.41,-0.023±0.24
IntegratedGradients,66.83±61.21,0.244±0.38,0.421±0.44,0.35±0.41,0.788±0.21,-0.029±0.23
